**Stage 1: ST-GCN gait gender classifier (model + training).**

Contains several successive training runs: a 10k-sample run (best val acc 67.50%), a 50k-sample run, the **30k-sample / 20-epoch run that produced the reported 72.65% validation accuracy** (24,000 train / 6,000 val; history in `results/training/training_history.csv`), and an interrupted resume-to-50-epochs run.

See the README's Implementation notes for where the training setup differs from the thesis text, and `Evaluate_GaitSTGCN.ipynb` for held-out test metrics.

In [ ]:
import os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch

# Check if CUDA (GPU support) is available
if torch.cuda.is_available():
    print("CUDA is available! Using GPU.")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("CUDA is not available. Using CPU.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
OPTIMIZED ST-GCN Pipeline for Fast Results
Key optimizations:
1. Uses annotations_present.csv (only existing files)
2. Subset training for quick results
3. Mixed precision training (2x speedup)
4. Efficient data loading
5. Progress tracking
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from tqdm import tqdm
import time

# ============================================================================
# GRAPH & MODEL
# ============================================================================

class Graph:
    """Skeleton graph structure for human body (18 joints OpenPose format)"""
    def __init__(self, num_node=18, layout='openpose'):
        self.num_node = num_node
        self.layout = layout
        self.get_edge()
        self.get_adjacency()

    def get_edge(self):
        """Define skeleton connectivity (bone connections)"""
        self.self_link = [(i, i) for i in range(self.num_node)]

        if self.num_node == 18:
            self.neighbor_link = [
                (0, 1), (1, 2), (2, 3), (3, 4), (1, 5), (5, 6), (6, 7),
                (1, 8), (8, 9), (9, 10), (10, 11), (8, 12), (12, 13),
                (13, 14), (0, 15), (15, 17), (0, 16), (16, 18),
            ]
        else:
            self.neighbor_link = [(i, i+1) for i in range(self.num_node-1)]

        self.edge = self.self_link + self.neighbor_link
        self.center = 1

    def get_adjacency(self):
        """Build normalized adjacency matrix"""
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edge:
            if i < self.num_node and j < self.num_node:
                A[i, j] = 1
                A[j, i] = 1

        A_with_self = A + np.eye(self.num_node)
        D = np.sum(A_with_self, axis=1)
        D_inv_sqrt = np.power(D, -0.5)
        D_inv_sqrt[np.isinf(D_inv_sqrt)] = 0
        D_mat = np.diag(D_inv_sqrt)
        A_normalized = D_mat @ A_with_self @ D_mat

        self.A = np.stack([np.eye(self.num_node), A_normalized, A_normalized])


class ConvTemporalGraphical(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv2d(in_channels, out_channels * kernel_size, kernel_size=1)

    def forward(self, x, A):
        x = self.conv(x)
        n, kc, t, v = x.size()
        x = x.view(n, self.kernel_size, kc // self.kernel_size, t, v)
        x = torch.einsum('nkctv,kvw->nctw', (x, A))
        return x.contiguous(), A


class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 dropout=0.2, residual=True):
        super().__init__()

        assert len(kernel_size) == 2
        assert kernel_size[0] % 2 == 1
        padding = ((kernel_size[0] - 1) // 2, 0)

        self.gcn = ConvTemporalGraphical(in_channels, out_channels, kernel_size[1])

        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, (kernel_size[0], 1),
                     (stride, 1), padding),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout, inplace=True),
        )

        if not residual:
            self.residual = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, A):
        res = self.residual(x)
        x, A = self.gcn(x, A)
        x = self.tcn(x) + res
        return self.relu(x), A


class GaitSTGCN(nn.Module):
    def __init__(self, in_channels=3, num_class=1, num_joints=18,
                 edge_importance_weighting=True, dropout=0.3):
        super().__init__()

        self.graph = Graph(num_node=num_joints)
        A = torch.tensor(self.graph.A, dtype=torch.float32, requires_grad=False)
        self.register_buffer('A', A)

        spatial_kernel_size = A.size(0)
        temporal_kernel_size = 9
        kernel_size = (temporal_kernel_size, spatial_kernel_size)

        self.data_bn = nn.BatchNorm1d(in_channels * A.size(1))

        self.st_gcn_networks = nn.ModuleList([
            STGCNBlock(in_channels, 64, kernel_size, 1, residual=False, dropout=0),
            STGCNBlock(64, 64, kernel_size, 1, dropout=dropout),
            STGCNBlock(64, 64, kernel_size, 1, dropout=dropout),
            STGCNBlock(64, 128, kernel_size, 2, dropout=dropout),
            STGCNBlock(128, 128, kernel_size, 1, dropout=dropout),
            STGCNBlock(128, 256, kernel_size, 2, dropout=dropout),
            STGCNBlock(256, 256, kernel_size, 1, dropout=dropout),
        ])

        if edge_importance_weighting:
            self.edge_importance = nn.Parameter(
                torch.ones(len(self.st_gcn_networks), *A.size())
            )
        else:
            self.edge_importance = [1] * len(self.st_gcn_networks)

        self.fc = nn.Linear(256, num_class)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        N, C, T, V, M = x.size()

        x = x.permute(0, 4, 3, 1, 2).contiguous()
        x = x.view(N * M, V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T)
        x = x.permute(0, 1, 3, 4, 2).contiguous()
        x = x.view(N * M, C, T, V)

        for gcn, importance in zip(self.st_gcn_networks, self.edge_importance):
            x, _ = gcn(x, self.A * importance)

        x = F.avg_pool2d(x, x.size()[2:])
        x = x.view(N, M, -1, 1, 1).mean(dim=1)
        x = x.view(N, -1)

        # Return raw logits (no sigmoid)
        # BCEWithLogitsLoss will apply sigmoid internally
        x = self.fc(x)
        return x

# ============================================================================
# OPTIMIZED PREPROCESSING
# ============================================================================

def detect_and_convert_format(data, target_V=18, target_C=3):
    shape = data.shape

    if len(shape) == 3:
        T, V, C = shape
        if V == target_V and C == target_C:
            return data
        elif V == target_V and C == 2 and target_C == 3:
            confidence = np.ones((T, V, 1), dtype=data.dtype)
            return np.concatenate([data, confidence], axis=2)
        flat = data.reshape(T, -1)
        return flat.reshape(T, target_V, target_C)

    elif len(shape) == 2:
        T, VC = shape
        if VC == target_V * target_C:
            return data.reshape(T, target_V, target_C)
        elif VC == target_V * 2 and target_C == 3:
            data_2d = data.reshape(T, target_V, 2)
            confidence = np.ones((T, target_V, 1), dtype=data.dtype)
            return np.concatenate([data_2d, confidence], axis=2)
        else:
            raise ValueError(f"Cannot convert shape {shape} to ({T}, {target_V}, {target_C})")
    else:
        raise ValueError(f"Unexpected data shape: {shape}")


def normalize_pose(data, method='center'):
    if method == 'center':
        center_idx = 1 if data.shape[1] >= 18 else 0
        center = data[:, center_idx:center_idx+1, :2]
        data_norm = data.copy()
        data_norm[:, :, :2] = data[:, :, :2] - center
        return data_norm
    return data


def handle_missing_values(data):
    for v in range(data.shape[1]):
        for c in range(data.shape[2]):
            series = data[:, v, c]
            nonzero = np.where(series != 0)[0]

            if len(nonzero) > 1:
                zero_indices = np.where(series == 0)[0]
                if len(zero_indices) > 0:
                    data[zero_indices, v, c] = np.interp(
                        zero_indices, nonzero, series[nonzero]
                    )
    return data

# ============================================================================
# OPTIMIZED DATASET
# ============================================================================

class GaitDatasetSTGCN(Dataset):
    def __init__(self, annotations_path, poses_dir, num_joints=18, num_coords=3,
                 normalize=True, handle_missing=True, skip_validation=True):
        self.annotations = pd.read_csv(annotations_path)
        self.poses_dir = poses_dir
        self.num_joints = num_joints
        self.num_coords = num_coords
        self.normalize = normalize
        self.handle_missing = handle_missing
        self.corrupted_indices = set()
        self.skip_validation = skip_validation

        self.file_names = self.annotations['file_name']
        self.labels = (self.annotations['Female'] > 0.5).astype(int).values

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        # Skip if we know this index is corrupted
        if idx in self.corrupted_indices:
            return self.__getitem__((idx + 1) % len(self))

        try:
            pose_path = os.path.join(self.poses_dir, self.file_names.iloc[idx])
            data = np.load(pose_path)

            # Validate data is not empty
            if data.size == 0:
                raise ValueError("Empty file")

            data = detect_and_convert_format(data, self.num_joints, self.num_coords)

            if self.handle_missing:
                data = handle_missing_values(data)
            if self.normalize:
                data = normalize_pose(data, method='center')

            pose_sequence = torch.from_numpy(data).float()
            pose_sequence = pose_sequence.permute(2, 0, 1).unsqueeze(-1)
            label = torch.tensor(self.labels[idx]).float()

            return pose_sequence, label

        except Exception as e:
            # Mark as corrupted and skip
            self.corrupted_indices.add(idx)
            # Return next valid sample
            return self.__getitem__((idx + 1) % len(self))


def collate_fn_stgcn(batch):
    sequences, labels = zip(*batch)
    max_T = max(seq.shape[1] for seq in sequences)

    padded_sequences = []
    lengths = []

    for seq in sequences:
        C, T, V, M = seq.shape
        lengths.append(T)

        if T < max_T:
            padding = torch.zeros(C, max_T - T, V, M)
            seq = torch.cat([seq, padding], dim=1)

        padded_sequences.append(seq)

    padded_sequences = torch.stack(padded_sequences, dim=0)
    labels = torch.stack(labels, dim=0)

    return padded_sequences, labels, torch.tensor(lengths)

# ============================================================================
# OPTIMIZED TRAINING FUNCTIONS
# ============================================================================

def train_epoch(model, dataloader, criterion, optimizer, device, scaler, use_amp=True):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Training", leave=False)
    for poses, labels, lengths in pbar:
        poses = poses.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast('cuda'):  # Updated deprecation warning
                outputs = model(poses)
                loss = criterion(outputs.squeeze(), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(poses)
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        # Apply sigmoid for predictions since model returns logits
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix({
            'loss': f'{total_loss/(pbar.n+1):.4f}',
            'acc': f'{100*correct/total:.2f}%'
        })

    return total_loss / len(dataloader), 100 * correct / total


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for poses, labels, lengths in tqdm(dataloader, desc="Validating", leave=False):
            poses = poses.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(poses)
            loss = criterion(outputs.squeeze(), labels)

            total_loss += loss.item()
            # Apply sigmoid for predictions since model returns logits
            predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(dataloader), 100 * correct / total

# ============================================================================
# MAIN OPTIMIZED PIPELINE
# ============================================================================

def main():
    print("=" * 80)
    print("OPTIMIZED ST-GCN GENDER CLASSIFICATION")
    print("=" * 80)

    # ========================================================================
    # CONFIGURATION - ADJUST THESE FOR SPEED
    # ========================================================================
    BASE_DIR = '/content/drive/MyDrive/Gait Project/DenseGait'

    # CRITICAL: Use annotations_present.csv (only existing files)
    ANNOTATIONS_PATH = os.path.join(BASE_DIR, 'annotations_present.csv')
    POSES_DIR = os.path.join(BASE_DIR, 'poses')
    OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

    # SPEED OPTIMIZATIONS
    USE_SUBSET = True        # Set False to use all data
    SUBSET_SIZE = 10000      # 10k samples = 1-2 hour training
    BATCH_SIZE = 32          # Increase to 64 if GPU has memory
    NUM_EPOCHS = 10          # 10 epochs for quick results
    NUM_WORKERS = 2          # Parallel data loading
    USE_AMP = True           # Mixed precision (2x speedup on A100)
    SKIP_VALIDATION = True   # Skip pre-validation (handle errors during training)

    # Model parameters
    NUM_JOINTS = 18
    NUM_COORDS = 3
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 0.0001
    DROPOUT = 0.3

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ========================================================================
    # 1. LOAD DATASET
    # ========================================================================
    print("\n1. Loading dataset...")
    print(f"   Using: {ANNOTATIONS_PATH}")

    if not os.path.exists(ANNOTATIONS_PATH):
        print(f"\n   ERROR: {ANNOTATIONS_PATH} not found!")
        print("   Run the file splitting script first to create annotations_present.csv")
        return

    dataset = GaitDatasetSTGCN(
        annotations_path=ANNOTATIONS_PATH,
        poses_dir=POSES_DIR,
        num_joints=NUM_JOINTS,
        num_coords=NUM_COORDS,
        normalize=True,
        handle_missing=True,
        skip_validation=SKIP_VALIDATION
    )
    print(f"   Total available samples: {len(dataset):,}")

    if SKIP_VALIDATION:
        print("   ⚡ Skipping pre-validation (errors handled during training)")

    # Use subset for faster experimentation
    if USE_SUBSET and len(dataset) > SUBSET_SIZE:
        indices = torch.randperm(len(dataset))[:SUBSET_SIZE].tolist()
        dataset = Subset(dataset, indices)
        print(f"   Using subset: {len(dataset):,} samples")

    # Split dataset
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(
        dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    print(f"   Training samples: {train_size:,}")
    print(f"   Validation samples: {val_size:,}")

    # ========================================================================
    # 2. CREATE OPTIMIZED DATALOADERS
    # ========================================================================
    print("\n2. Creating dataloaders...")
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn_stgcn,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn_stgcn,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0
    )
    print(f"   Batch size: {BATCH_SIZE}")
    print(f"   Batches per epoch: {len(train_loader)}")

    # ========================================================================
    # 3. INITIALIZE MODEL
    # ========================================================================
    print("\n3. Initializing model...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"   Device: {device}")

    if device.type == 'cpu':
        print("   WARNING: Running on CPU. Training will be SLOW!")
        print("   Go to Runtime > Change runtime type > Select GPU")

    model = GaitSTGCN(
        in_channels=NUM_COORDS,
        num_class=1,
        num_joints=NUM_JOINTS,
        edge_importance_weighting=True,
        dropout=DROPOUT
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"   Total parameters: {total_params:,}")

    # Loss, optimizer, and mixed precision scaler
    criterion = nn.BCEWithLogitsLoss()  # Safe for autocast (combines sigmoid + BCE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=5, gamma=0.5
    )
    scaler = torch.amp.GradScaler('cuda') if USE_AMP else None  # Updated API

    if USE_AMP:
        print(f"   Mixed precision: ENABLED (2x speedup)")

    # ========================================================================
    # 4. TRAINING LOOP
    # ========================================================================
    print("\n4. Training...")
    print("-" * 80)

    # Create subdirectory for epoch checkpoints
    checkpoints_dir = os.path.join(OUTPUT_DIR, 'checkpoints')
    os.makedirs(checkpoints_dir, exist_ok=True)

    best_val_acc = 0.0
    training_history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    start_time = time.time()

    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()

        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device, scaler, USE_AMP
        )
        val_loss, val_acc = validate(
            model, val_loader, criterion, device
        )

        scheduler.step()

        epoch_time = time.time() - epoch_start

        training_history['train_loss'].append(train_loss)
        training_history['train_acc'].append(train_acc)
        training_history['val_loss'].append(val_loss)
        training_history['val_acc'].append(val_acc)

        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} ({epoch_time:.1f}s)")
        print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
        print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

        # Save checkpoint for this epoch
        epoch_checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'training_history': training_history,
        }

        epoch_path = os.path.join(checkpoints_dir, f'model_epoch_{epoch+1}.pth')
        torch.save(epoch_checkpoint, epoch_path)
        print(f"  💾 Checkpoint saved: model_epoch_{epoch+1}.pth")

        # Save best model (separate from epoch checkpoints)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
                'train_acc': train_acc,
                'train_loss': train_loss,
            }
            torch.save(best_checkpoint, os.path.join(OUTPUT_DIR, 'best_model.pth'))
            print(f"  ✓ Best model updated! (Val Acc: {val_acc:.2f}%)")

    total_time = time.time() - start_time

    print("\n" + "=" * 80)
    print("TRAINING COMPLETE!")
    print("=" * 80)
    print(f"Total time: {total_time/60:.1f} minutes")
    print(f"Best validation accuracy: {best_val_acc:.2f}%")

    # Save corrupted files encountered during training
    if len(dataset.corrupted_indices) > 0:
        print(f"\n⚠️  Corrupted files encountered: {len(dataset.corrupted_indices)}")
        corrupted_files = [dataset.file_names.iloc[i] for i in dataset.corrupted_indices]
        corrupted_annotations = dataset.annotations[
            dataset.annotations['file_name'].isin(corrupted_files)
        ][['Female', 'file_name']]

        skipped_path = os.path.join(BASE_DIR, 'annotations_skipped_in_training.csv')
        corrupted_annotations.to_csv(skipped_path, index=False)
        print(f"   ✓ Skipped files saved to: {skipped_path}")

    # Save results
    torch.save({
        'model_state_dict': model.state_dict(),
        'training_history': training_history,
    }, os.path.join(OUTPUT_DIR, 'final_model.pth'))

    history_df = pd.DataFrame(training_history)
    history_df.to_csv(os.path.join(OUTPUT_DIR, 'training_history.csv'), index=False)

    print(f"\n✓ All outputs saved to: {OUTPUT_DIR}")
    print("=" * 80)


if __name__ == '__main__':
    main()

The above saved best model for 10k samples. It serves well for testing and producing results. now lets continue for 50k dataset.

In [ ]:
"""
OPTIMIZED ST-GCN Pipeline for Fast Results
Key optimizations:
1. Uses annotations_present.csv (only existing files)
2. Subset training for quick results
3. Mixed precision training (2x speedup)
4. Efficient data loading
5. Progress tracking
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from tqdm import tqdm
import time

# ============================================================================
# GRAPH & MODEL
# ============================================================================

class Graph:
    """Skeleton graph structure for human body (18 joints OpenPose format)"""
    def __init__(self, num_node=18, layout='openpose'):
        self.num_node = num_node
        self.layout = layout
        self.get_edge()
        self.get_adjacency()

    def get_edge(self):
        """Define skeleton connectivity (bone connections)"""
        self.self_link = [(i, i) for i in range(self.num_node)]

        if self.num_node == 18:
            self.neighbor_link = [
                (0, 1), (1, 2), (2, 3), (3, 4), (1, 5), (5, 6), (6, 7),
                (1, 8), (8, 9), (9, 10), (10, 11), (8, 12), (12, 13),
                (13, 14), (0, 15), (15, 17), (0, 16), (16, 18),
            ]
        else:
            self.neighbor_link = [(i, i+1) for i in range(self.num_node-1)]

        self.edge = self.self_link + self.neighbor_link
        self.center = 1

    def get_adjacency(self):
        """Build normalized adjacency matrix"""
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edge:
            if i < self.num_node and j < self.num_node:
                A[i, j] = 1
                A[j, i] = 1

        A_with_self = A + np.eye(self.num_node)
        D = np.sum(A_with_self, axis=1)
        D_inv_sqrt = np.power(D, -0.5)
        D_inv_sqrt[np.isinf(D_inv_sqrt)] = 0
        D_mat = np.diag(D_inv_sqrt)
        A_normalized = D_mat @ A_with_self @ D_mat

        self.A = np.stack([np.eye(self.num_node), A_normalized, A_normalized])


class ConvTemporalGraphical(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv2d(in_channels, out_channels * kernel_size, kernel_size=1)

    def forward(self, x, A):
        x = self.conv(x)
        n, kc, t, v = x.size()
        x = x.view(n, self.kernel_size, kc // self.kernel_size, t, v)
        x = torch.einsum('nkctv,kvw->nctw', (x, A))
        return x.contiguous(), A


class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 dropout=0.2, residual=True):
        super().__init__()

        assert len(kernel_size) == 2
        assert kernel_size[0] % 2 == 1
        padding = ((kernel_size[0] - 1) // 2, 0)

        self.gcn = ConvTemporalGraphical(in_channels, out_channels, kernel_size[1])

        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, (kernel_size[0], 1),
                     (stride, 1), padding),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout, inplace=True),
        )

        if not residual:
            self.residual = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, A):
        res = self.residual(x)
        x, A = self.gcn(x, A)
        x = self.tcn(x) + res
        return self.relu(x), A


class GaitSTGCN(nn.Module):
    def __init__(self, in_channels=3, num_class=1, num_joints=18,
                 edge_importance_weighting=True, dropout=0.3):
        super().__init__()

        self.graph = Graph(num_node=num_joints)
        A = torch.tensor(self.graph.A, dtype=torch.float32, requires_grad=False)
        self.register_buffer('A', A)

        spatial_kernel_size = A.size(0)
        temporal_kernel_size = 9
        kernel_size = (temporal_kernel_size, spatial_kernel_size)

        self.data_bn = nn.BatchNorm1d(in_channels * A.size(1))

        self.st_gcn_networks = nn.ModuleList([
            STGCNBlock(in_channels, 64, kernel_size, 1, residual=False, dropout=0),
            STGCNBlock(64, 64, kernel_size, 1, dropout=dropout),
            STGCNBlock(64, 64, kernel_size, 1, dropout=dropout),
            STGCNBlock(64, 128, kernel_size, 2, dropout=dropout),
            STGCNBlock(128, 128, kernel_size, 1, dropout=dropout),
            STGCNBlock(128, 256, kernel_size, 2, dropout=dropout),
            STGCNBlock(256, 256, kernel_size, 1, dropout=dropout),
        ])

        if edge_importance_weighting:
            self.edge_importance = nn.Parameter(
                torch.ones(len(self.st_gcn_networks), *A.size())
            )
        else:
            self.edge_importance = [1] * len(self.st_gcn_networks)

        self.fc = nn.Linear(256, num_class)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        N, C, T, V, M = x.size()

        x = x.permute(0, 4, 3, 1, 2).contiguous()
        x = x.view(N * M, V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T)
        x = x.permute(0, 1, 3, 4, 2).contiguous()
        x = x.view(N * M, C, T, V)

        for gcn, importance in zip(self.st_gcn_networks, self.edge_importance):
            x, _ = gcn(x, self.A * importance)

        x = F.avg_pool2d(x, x.size()[2:])
        x = x.view(N, M, -1, 1, 1).mean(dim=1)
        x = x.view(N, -1)

        # Return raw logits (no sigmoid)
        # BCEWithLogitsLoss will apply sigmoid internally
        x = self.fc(x)
        return x

# ============================================================================
# OPTIMIZED PREPROCESSING
# ============================================================================

def detect_and_convert_format(data, target_V=18, target_C=3):
    shape = data.shape

    if len(shape) == 3:
        T, V, C = shape
        if V == target_V and C == target_C:
            return data
        elif V == target_V and C == 2 and target_C == 3:
            confidence = np.ones((T, V, 1), dtype=data.dtype)
            return np.concatenate([data, confidence], axis=2)
        flat = data.reshape(T, -1)
        return flat.reshape(T, target_V, target_C)

    elif len(shape) == 2:
        T, VC = shape
        if VC == target_V * target_C:
            return data.reshape(T, target_V, target_C)
        elif VC == target_V * 2 and target_C == 3:
            data_2d = data.reshape(T, target_V, 2)
            confidence = np.ones((T, target_V, 1), dtype=data.dtype)
            return np.concatenate([data_2d, confidence], axis=2)
        else:
            raise ValueError(f"Cannot convert shape {shape} to ({T}, {target_V}, {target_C})")
    else:
        raise ValueError(f"Unexpected data shape: {shape}")


def normalize_pose(data, method='center'):
    if method == 'center':
        center_idx = 1 if data.shape[1] >= 18 else 0
        center = data[:, center_idx:center_idx+1, :2]
        data_norm = data.copy()
        data_norm[:, :, :2] = data[:, :, :2] - center
        return data_norm
    return data


def handle_missing_values(data):
    for v in range(data.shape[1]):
        for c in range(data.shape[2]):
            series = data[:, v, c]
            nonzero = np.where(series != 0)[0]

            if len(nonzero) > 1:
                zero_indices = np.where(series == 0)[0]
                if len(zero_indices) > 0:
                    data[zero_indices, v, c] = np.interp(
                        zero_indices, nonzero, series[nonzero]
                    )
    return data

# ============================================================================
# OPTIMIZED DATASET
# ============================================================================

class GaitDatasetSTGCN(Dataset):
    def __init__(self, annotations_path, poses_dir, num_joints=18, num_coords=3,
                 normalize=True, handle_missing=True, skip_validation=True):
        self.annotations = pd.read_csv(annotations_path)
        self.poses_dir = poses_dir
        self.num_joints = num_joints
        self.num_coords = num_coords
        self.normalize = normalize
        self.handle_missing = handle_missing
        self.corrupted_indices = set()
        self.skip_validation = skip_validation

        self.file_names = self.annotations['file_name']
        self.labels = (self.annotations['Female'] > 0.5).astype(int).values

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        # Skip if we know this index is corrupted
        if idx in self.corrupted_indices:
            return self.__getitem__((idx + 1) % len(self))

        try:
            pose_path = os.path.join(self.poses_dir, self.file_names.iloc[idx])
            data = np.load(pose_path)

            # Validate data is not empty
            if data.size == 0:
                raise ValueError("Empty file")

            data = detect_and_convert_format(data, self.num_joints, self.num_coords)

            if self.handle_missing:
                data = handle_missing_values(data)
            if self.normalize:
                data = normalize_pose(data, method='center')

            pose_sequence = torch.from_numpy(data).float()
            pose_sequence = pose_sequence.permute(2, 0, 1).unsqueeze(-1)
            label = torch.tensor(self.labels[idx]).float()

            return pose_sequence, label

        except Exception as e:
            # Mark as corrupted and skip
            self.corrupted_indices.add(idx)
            # Return next valid sample
            return self.__getitem__((idx + 1) % len(self))


def collate_fn_stgcn(batch):
    sequences, labels = zip(*batch)
    max_T = max(seq.shape[1] for seq in sequences)

    padded_sequences = []
    lengths = []

    for seq in sequences:
        C, T, V, M = seq.shape
        lengths.append(T)

        if T < max_T:
            padding = torch.zeros(C, max_T - T, V, M)
            seq = torch.cat([seq, padding], dim=1)

        padded_sequences.append(seq)

    padded_sequences = torch.stack(padded_sequences, dim=0)
    labels = torch.stack(labels, dim=0)

    return padded_sequences, labels, torch.tensor(lengths)

# ============================================================================
# OPTIMIZED TRAINING FUNCTIONS
# ============================================================================

def train_epoch(model, dataloader, criterion, optimizer, device, scaler, use_amp=True):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Training", leave=False)
    for poses, labels, lengths in pbar:
        poses = poses.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast('cuda'):  # Updated deprecation warning
                outputs = model(poses)
                loss = criterion(outputs.squeeze(), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(poses)
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        # Apply sigmoid for predictions since model returns logits
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix({
            'loss': f'{total_loss/(pbar.n+1):.4f}',
            'acc': f'{100*correct/total:.2f}%'
        })

    return total_loss / len(dataloader), 100 * correct / total


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for poses, labels, lengths in tqdm(dataloader, desc="Validating", leave=False):
            poses = poses.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(poses)
            loss = criterion(outputs.squeeze(), labels)

            total_loss += loss.item()
            # Apply sigmoid for predictions since model returns logits
            predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(dataloader), 100 * correct / total

# ============================================================================
# MAIN OPTIMIZED PIPELINE
# ============================================================================

def main():
    print("=" * 80)
    print("OPTIMIZED ST-GCN GENDER CLASSIFICATION")
    print("=" * 80)

    # ========================================================================
    # CONFIGURATION - ADJUST THESE FOR SPEED
    # ========================================================================
    BASE_DIR = '/content/drive/MyDrive/Gait Project/DenseGait'

    # CRITICAL: Use annotations_present.csv (only existing files)
    ANNOTATIONS_PATH = os.path.join(BASE_DIR, 'annotations_present.csv')
    POSES_DIR = os.path.join(BASE_DIR, 'poses')
    OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

    # SPEED OPTIMIZATIONS - OPTIMIZED FOR GOOGLE DRIVE
    USE_SUBSET = True        # *** Use subset to avoid Drive slowdown ***
    SUBSET_SIZE = 50000      # *** 50k samples = good balance ***
    BATCH_SIZE = 128         # *** LARGER batches = fewer I/O calls ***
    NUM_EPOCHS = 20          # 20 epochs for good accuracy
    NUM_WORKERS = 4          # *** MORE workers for parallel loading ***
    USE_AMP = True           # Mixed precision (2x speedup on A100)
    SKIP_VALIDATION = True   # Skip pre-validation (handle errors during training)
    PREFETCH_FACTOR = 4      # *** Prefetch batches to hide I/O latency ***

    # Model parameters
    NUM_JOINTS = 18
    NUM_COORDS = 3
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 0.0001
    DROPOUT = 0.3

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ========================================================================
    # 1. LOAD DATASET
    # ========================================================================
    print("\n1. Loading dataset...")
    print(f"   Using: {ANNOTATIONS_PATH}")

    if not os.path.exists(ANNOTATIONS_PATH):
        print(f"\n   ERROR: {ANNOTATIONS_PATH} not found!")
        print("   Run the file splitting script first to create annotations_present.csv")
        return

    dataset = GaitDatasetSTGCN(
        annotations_path=ANNOTATIONS_PATH,
        poses_dir=POSES_DIR,
        num_joints=NUM_JOINTS,
        num_coords=NUM_COORDS,
        normalize=True,
        handle_missing=True,
        skip_validation=SKIP_VALIDATION
    )
    print(f"   Total available samples: {len(dataset):,}")

    if SKIP_VALIDATION:
        print("   ⚡ Skipping pre-validation (errors handled during training)")

    # Use subset for faster experimentation
    if USE_SUBSET and len(dataset) > SUBSET_SIZE:
        indices = torch.randperm(len(dataset))[:SUBSET_SIZE].tolist()
        dataset = Subset(dataset, indices)
        print(f"   Using subset: {len(dataset):,} samples")

    # Split dataset
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(
        dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    print(f"   Training samples: {train_size:,}")
    print(f"   Validation samples: {val_size:,}")

    # ========================================================================
    # 2. CREATE OPTIMIZED DATALOADERS
    # ========================================================================
    print("\n2. Creating dataloaders...")
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn_stgcn,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn_stgcn,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
    )
    print(f"   Batch size: {BATCH_SIZE}")
    print(f"   Batches per epoch: {len(train_loader)}")
    print(f"   Workers: {NUM_WORKERS}, Prefetch: {PREFETCH_FACTOR}")

    # ========================================================================
    # 3. INITIALIZE MODEL
    # ========================================================================
    print("\n3. Initializing model...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"   Device: {device}")

    if device.type == 'cpu':
        print("   WARNING: Running on CPU. Training will be SLOW!")
        print("   Go to Runtime > Change runtime type > Select GPU")

    model = GaitSTGCN(
        in_channels=NUM_COORDS,
        num_class=1,
        num_joints=NUM_JOINTS,
        edge_importance_weighting=True,
        dropout=DROPOUT
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"   Total parameters: {total_params:,}")

    # Loss, optimizer, and mixed precision scaler
    criterion = nn.BCEWithLogitsLoss()  # Safe for autocast (combines sigmoid + BCE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=5, gamma=0.5
    )
    scaler = torch.amp.GradScaler('cuda') if USE_AMP else None  # Updated API

    if USE_AMP:
        print(f"   Mixed precision: ENABLED (2x speedup)")

    # ========================================================================
    # 4. TRAINING LOOP
    # ========================================================================
    print("\n4. Training...")
    print("-" * 80)

    # Create subdirectory for epoch checkpoints
    checkpoints_dir = os.path.join(OUTPUT_DIR, 'checkpoints')
    os.makedirs(checkpoints_dir, exist_ok=True)

    best_val_acc = 0.0
    training_history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    start_time = time.time()

    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()

        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device, scaler, USE_AMP
        )
        val_loss, val_acc = validate(
            model, val_loader, criterion, device
        )

        scheduler.step()

        epoch_time = time.time() - epoch_start

        training_history['train_loss'].append(train_loss)
        training_history['train_acc'].append(train_acc)
        training_history['val_loss'].append(val_loss)
        training_history['val_acc'].append(val_acc)

        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} ({epoch_time:.1f}s)")
        print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
        print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

        # Save checkpoint for this epoch
        epoch_checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'training_history': training_history,
        }

        epoch_path = os.path.join(checkpoints_dir, f'model_epoch_{epoch+1}.pth')
        torch.save(epoch_checkpoint, epoch_path)
        print(f"  💾 Checkpoint saved: model_epoch_{epoch+1}.pth")

        # Save best model (separate from epoch checkpoints)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
                'train_acc': train_acc,
                'train_loss': train_loss,
            }
            torch.save(best_checkpoint, os.path.join(OUTPUT_DIR, 'best_model.pth'))
            print(f"  ✓ Best model updated! (Val Acc: {val_acc:.2f}%)")

    total_time = time.time() - start_time

    print("\n" + "=" * 80)
    print("TRAINING COMPLETE!")
    print("=" * 80)
    print(f"Total time: {total_time/60:.1f} minutes")
    print(f"Best validation accuracy: {best_val_acc:.2f}%")

    # Save corrupted files encountered during training
    # Handle both Subset and regular dataset
    base_dataset = dataset.dataset if isinstance(dataset, Subset) else dataset
    if hasattr(base_dataset, 'corrupted_indices') and len(base_dataset.corrupted_indices) > 0:
        print(f"\n⚠️  Corrupted files encountered: {len(base_dataset.corrupted_indices)}")
        corrupted_files = [base_dataset.file_names.iloc[i] for i in base_dataset.corrupted_indices]
        corrupted_annotations = base_dataset.annotations[
            base_dataset.annotations['file_name'].isin(corrupted_files)
        ][['Female', 'file_name']]

        skipped_path = os.path.join(BASE_DIR, 'annotations_skipped_in_training.csv')
        corrupted_annotations.to_csv(skipped_path, index=False)
        print(f"   ✓ Skipped files saved to: {skipped_path}")

    # Save results
    torch.save({
        'model_state_dict': model.state_dict(),
        'training_history': training_history,
    }, os.path.join(OUTPUT_DIR, 'final_model.pth'))

    history_df = pd.DataFrame(training_history)
    history_df.to_csv(os.path.join(OUTPUT_DIR, 'training_history.csv'), index=False)

    print(f"\n✓ All outputs saved to: {OUTPUT_DIR}")
    print("=" * 80)


if __name__ == '__main__':
    main()

50K didn't run lets do 30k with 64 batch size

In [ ]:
"""
OPTIMIZED ST-GCN Pipeline for Fast Results
Key optimizations:
1. Uses annotations_present.csv (only existing files)
2. Subset training for quick results
3. Mixed precision training (2x speedup)
4. Efficient data loading
5. Progress tracking
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from tqdm import tqdm
import time

# ============================================================================
# GRAPH & MODEL (Your existing code - unchanged)
# ============================================================================

class Graph:
    """Skeleton graph structure for human body (18 joints OpenPose format)"""
    def __init__(self, num_node=18, layout='openpose'):
        self.num_node = num_node
        self.layout = layout
        self.get_edge()
        self.get_adjacency()

    def get_edge(self):
        """Define skeleton connectivity (bone connections)"""
        self.self_link = [(i, i) for i in range(self.num_node)]

        if self.num_node == 18:
            self.neighbor_link = [
                (0, 1), (1, 2), (2, 3), (3, 4), (1, 5), (5, 6), (6, 7),
                (1, 8), (8, 9), (9, 10), (10, 11), (8, 12), (12, 13),
                (13, 14), (0, 15), (15, 17), (0, 16), (16, 18),
            ]
        else:
            self.neighbor_link = [(i, i+1) for i in range(self.num_node-1)]

        self.edge = self.self_link + self.neighbor_link
        self.center = 1

    def get_adjacency(self):
        """Build normalized adjacency matrix"""
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edge:
            if i < self.num_node and j < self.num_node:
                A[i, j] = 1
                A[j, i] = 1

        A_with_self = A + np.eye(self.num_node)
        D = np.sum(A_with_self, axis=1)
        D_inv_sqrt = np.power(D, -0.5)
        D_inv_sqrt[np.isinf(D_inv_sqrt)] = 0
        D_mat = np.diag(D_inv_sqrt)
        A_normalized = D_mat @ A_with_self @ D_mat

        self.A = np.stack([np.eye(self.num_node), A_normalized, A_normalized])


class ConvTemporalGraphical(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv2d(in_channels, out_channels * kernel_size, kernel_size=1)

    def forward(self, x, A):
        x = self.conv(x)
        n, kc, t, v = x.size()
        x = x.view(n, self.kernel_size, kc // self.kernel_size, t, v)
        x = torch.einsum('nkctv,kvw->nctw', (x, A))
        return x.contiguous(), A


class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 dropout=0.2, residual=True):
        super().__init__()

        assert len(kernel_size) == 2
        assert kernel_size[0] % 2 == 1
        padding = ((kernel_size[0] - 1) // 2, 0)

        self.gcn = ConvTemporalGraphical(in_channels, out_channels, kernel_size[1])

        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, (kernel_size[0], 1),
                     (stride, 1), padding),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout, inplace=True),
        )

        if not residual:
            self.residual = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, A):
        res = self.residual(x)
        x, A = self.gcn(x, A)
        x = self.tcn(x) + res
        return self.relu(x), A


class GaitSTGCN(nn.Module):
    def __init__(self, in_channels=3, num_class=1, num_joints=18,
                 edge_importance_weighting=True, dropout=0.3):
        super().__init__()

        self.graph = Graph(num_node=num_joints)
        A = torch.tensor(self.graph.A, dtype=torch.float32, requires_grad=False)
        self.register_buffer('A', A)

        spatial_kernel_size = A.size(0)
        temporal_kernel_size = 9
        kernel_size = (temporal_kernel_size, spatial_kernel_size)

        self.data_bn = nn.BatchNorm1d(in_channels * A.size(1))

        self.st_gcn_networks = nn.ModuleList([
            STGCNBlock(in_channels, 64, kernel_size, 1, residual=False, dropout=0),
            STGCNBlock(64, 64, kernel_size, 1, dropout=dropout),
            STGCNBlock(64, 64, kernel_size, 1, dropout=dropout),
            STGCNBlock(64, 128, kernel_size, 2, dropout=dropout),
            STGCNBlock(128, 128, kernel_size, 1, dropout=dropout),
            STGCNBlock(128, 256, kernel_size, 2, dropout=dropout),
            STGCNBlock(256, 256, kernel_size, 1, dropout=dropout),
        ])

        if edge_importance_weighting:
            self.edge_importance = nn.Parameter(
                torch.ones(len(self.st_gcn_networks), *A.size())
            )
        else:
            self.edge_importance = [1] * len(self.st_gcn_networks)

        self.fc = nn.Linear(256, num_class)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        N, C, T, V, M = x.size()

        x = x.permute(0, 4, 3, 1, 2).contiguous()
        x = x.view(N * M, V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T)
        x = x.permute(0, 1, 3, 4, 2).contiguous()
        x = x.view(N * M, C, T, V)

        for gcn, importance in zip(self.st_gcn_networks, self.edge_importance):
            x, _ = gcn(x, self.A * importance)

        x = F.avg_pool2d(x, x.size()[2:])
        x = x.view(N, M, -1, 1, 1).mean(dim=1)
        x = x.view(N, -1)

        # Return raw logits (no sigmoid)
        # BCEWithLogitsLoss will apply sigmoid internally
        x = self.fc(x)
        return x

# ============================================================================
# OPTIMIZED PREPROCESSING
# ============================================================================

def detect_and_convert_format(data, target_V=18, target_C=3):
    shape = data.shape

    if len(shape) == 3:
        T, V, C = shape
        if V == target_V and C == target_C:
            return data
        elif V == target_V and C == 2 and target_C == 3:
            confidence = np.ones((T, V, 1), dtype=data.dtype)
            return np.concatenate([data, confidence], axis=2)
        flat = data.reshape(T, -1)
        return flat.reshape(T, target_V, target_C)

    elif len(shape) == 2:
        T, VC = shape
        if VC == target_V * target_C:
            return data.reshape(T, target_V, target_C)
        elif VC == target_V * 2 and target_C == 3:
            data_2d = data.reshape(T, target_V, 2)
            confidence = np.ones((T, target_V, 1), dtype=data.dtype)
            return np.concatenate([data_2d, confidence], axis=2)
        else:
            raise ValueError(f"Cannot convert shape {shape} to ({T}, {target_V}, {target_C})")
    else:
        raise ValueError(f"Unexpected data shape: {shape}")


def normalize_pose(data, method='center'):
    if method == 'center':
        center_idx = 1 if data.shape[1] >= 18 else 0
        center = data[:, center_idx:center_idx+1, :2]
        data_norm = data.copy()
        data_norm[:, :, :2] = data[:, :, :2] - center
        return data_norm
    return data


def handle_missing_values(data):
    for v in range(data.shape[1]):
        for c in range(data.shape[2]):
            series = data[:, v, c]
            nonzero = np.where(series != 0)[0]

            if len(nonzero) > 1:
                zero_indices = np.where(series == 0)[0]
                if len(zero_indices) > 0:
                    data[zero_indices, v, c] = np.interp(
                        zero_indices, nonzero, series[nonzero]
                    )
    return data

# ============================================================================
# OPTIMIZED DATASET
# ============================================================================

class GaitDatasetSTGCN(Dataset):
    def __init__(self, annotations_path, poses_dir, num_joints=18, num_coords=3,
                 normalize=True, handle_missing=True, skip_validation=True):
        self.annotations = pd.read_csv(annotations_path)
        self.poses_dir = poses_dir
        self.num_joints = num_joints
        self.num_coords = num_coords
        self.normalize = normalize
        self.handle_missing = handle_missing
        self.corrupted_indices = set()
        self.skip_validation = skip_validation

        self.file_names = self.annotations['file_name']
        self.labels = (self.annotations['Female'] > 0.5).astype(int).values

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        # Skip if we know this index is corrupted
        if idx in self.corrupted_indices:
            return self.__getitem__((idx + 1) % len(self))

        try:
            pose_path = os.path.join(self.poses_dir, self.file_names.iloc[idx])
            data = np.load(pose_path)

            # Validate data is not empty
            if data.size == 0:
                raise ValueError("Empty file")

            data = detect_and_convert_format(data, self.num_joints, self.num_coords)

            if self.handle_missing:
                data = handle_missing_values(data)
            if self.normalize:
                data = normalize_pose(data, method='center')

            pose_sequence = torch.from_numpy(data).float()
            pose_sequence = pose_sequence.permute(2, 0, 1).unsqueeze(-1)
            label = torch.tensor(self.labels[idx]).float()

            return pose_sequence, label

        except Exception as e:
            # Mark as corrupted and skip
            self.corrupted_indices.add(idx)
            # Return next valid sample
            return self.__getitem__((idx + 1) % len(self))


def collate_fn_stgcn(batch, max_seq_length=300):
    """Collate function that pads temporal dimension and truncates long sequences"""
    sequences, labels = zip(*batch)

    # Truncate sequences that are too long (memory safety)
    sequences = [seq[:, :max_seq_length, :, :] if seq.shape[1] > max_seq_length else seq
                 for seq in sequences]

    max_T = max(seq.shape[1] for seq in sequences)

    padded_sequences = []
    lengths = []

    for seq in sequences:
        C, T, V, M = seq.shape
        lengths.append(T)

        if T < max_T:
            padding = torch.zeros(C, max_T - T, V, M)
            seq = torch.cat([seq, padding], dim=1)

        padded_sequences.append(seq)

    padded_sequences = torch.stack(padded_sequences, dim=0)
    labels = torch.stack(labels, dim=0)

    return padded_sequences, labels, torch.tensor(lengths)

# ============================================================================
# OPTIMIZED TRAINING FUNCTIONS
# ============================================================================

def train_epoch(model, dataloader, criterion, optimizer, device, scaler, use_amp=True, clear_cache_interval=25):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Training", leave=False)
    for batch_idx, (poses, labels, lengths) in enumerate(pbar):
        poses = poses.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast('cuda'):
                outputs = model(poses)
                loss = criterion(outputs.squeeze(), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(poses)
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Clear cache periodically to prevent fragmentation
        if (batch_idx + 1) % clear_cache_interval == 0:
            torch.cuda.empty_cache()

        pbar.set_postfix({
            'loss': f'{total_loss/(batch_idx+1):.4f}',
            'acc': f'{100*correct/total:.2f}%'
        })

    return total_loss / len(dataloader), 100 * correct / total


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for poses, labels, lengths in tqdm(dataloader, desc="Validating", leave=False):
            poses = poses.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(poses)
            loss = criterion(outputs.squeeze(), labels)

            total_loss += loss.item()
            # Apply sigmoid for predictions since model returns logits
            predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(dataloader), 100 * correct / total

# ============================================================================
# MAIN OPTIMIZED PIPELINE
# ============================================================================

def main():
    print("=" * 80)
    print("OPTIMIZED ST-GCN GENDER CLASSIFICATION")
    print("=" * 80)

    # ========================================================================
    # BULLETPROOF CONFIG - GUARANTEED TO COMPLETE
    # ========================================================================
    BASE_DIR = '/content/drive/MyDrive/Gait Project/DenseGait'

    # CRITICAL: Use annotations_present.csv (only existing files)
    ANNOTATIONS_PATH = os.path.join(BASE_DIR, 'annotations_present.csv')
    POSES_DIR = os.path.join(BASE_DIR, 'poses')
    OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

    # SAFE CONFIGURATION - Will NOT run out of memory
    USE_SUBSET = True
    SUBSET_SIZE = 30000      # *** REDUCED from 50k for safety ***
    BATCH_SIZE = 64          # *** REDUCED from 128 - uses ~50GB GPU ***
    NUM_EPOCHS = 20
    NUM_WORKERS = 2          # *** REDUCED from 4 to save RAM ***
    USE_AMP = True
    SKIP_VALIDATION = True
    PREFETCH_FACTOR = 2      # *** REDUCED from 4 to save RAM ***

    # Memory management
    CLEAR_CACHE_INTERVAL = 25  # Clear GPU cache every 25 batches
    MAX_SEQUENCE_LENGTH = 300  # Truncate very long sequences

    # Model parameters
    NUM_JOINTS = 18
    NUM_COORDS = 3
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 0.0001
    DROPOUT = 0.3

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ========================================================================
    # 1. LOAD DATASET
    # ========================================================================
    print("\n1. Loading dataset...")
    print(f"   Using: {ANNOTATIONS_PATH}")

    if not os.path.exists(ANNOTATIONS_PATH):
        print(f"\n   ERROR: {ANNOTATIONS_PATH} not found!")
        print("   Run the file splitting script first to create annotations_present.csv")
        return

    dataset = GaitDatasetSTGCN(
        annotations_path=ANNOTATIONS_PATH,
        poses_dir=POSES_DIR,
        num_joints=NUM_JOINTS,
        num_coords=NUM_COORDS,
        normalize=True,
        handle_missing=True,
        skip_validation=SKIP_VALIDATION
    )
    print(f"   Total available samples: {len(dataset):,}")

    if SKIP_VALIDATION:
        print("   ⚡ Skipping pre-validation (errors handled during training)")

    # Use subset for faster experimentation
    if USE_SUBSET and len(dataset) > SUBSET_SIZE:
        indices = torch.randperm(len(dataset))[:SUBSET_SIZE].tolist()
        dataset = Subset(dataset, indices)
        print(f"   Using subset: {len(dataset):,} samples")

    # Split dataset
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(
        dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    print(f"   Training samples: {train_size:,}")
    print(f"   Validation samples: {val_size:,}")

    # ========================================================================
    # 2. CREATE OPTIMIZED DATALOADERS
    # ========================================================================
    print("\n2. Creating dataloaders...")

    # Custom collate function with max sequence length
    def collate_wrapper(batch):
        return collate_fn_stgcn(batch, max_seq_length=MAX_SEQUENCE_LENGTH)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_wrapper,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_wrapper,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
    )
    print(f"   Batch size: {BATCH_SIZE}")
    print(f"   Batches per epoch: {len(train_loader)}")
    print(f"   Workers: {NUM_WORKERS}, Prefetch: {PREFETCH_FACTOR}")
    print(f"   Max sequence length: {MAX_SEQUENCE_LENGTH} (truncated if longer)")

    # ========================================================================
    # 3. INITIALIZE MODEL
    # ========================================================================
    print("\n3. Initializing model...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"   Device: {device}")

    if device.type == 'cpu':
        print("   WARNING: Running on CPU. Training will be SLOW!")
        print("   Go to Runtime > Change runtime type > Select GPU")

    model = GaitSTGCN(
        in_channels=NUM_COORDS,
        num_class=1,
        num_joints=NUM_JOINTS,
        edge_importance_weighting=True,
        dropout=DROPOUT
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"   Total parameters: {total_params:,}")

    # Loss, optimizer, and mixed precision scaler
    criterion = nn.BCEWithLogitsLoss()  # Safe for autocast (combines sigmoid + BCE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=5, gamma=0.5
    )
    scaler = torch.amp.GradScaler('cuda') if USE_AMP else None  # Updated API

    if USE_AMP:
        print(f"   Mixed precision: ENABLED (2x speedup)")

    # ========================================================================
    # 4. TRAINING LOOP
    # ========================================================================
    print("\n4. Training...")
    print("-" * 80)

    # Create subdirectory for epoch checkpoints
    checkpoints_dir = os.path.join(OUTPUT_DIR, 'checkpoints')
    os.makedirs(checkpoints_dir, exist_ok=True)

    best_val_acc = 0.0
    training_history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    start_time = time.time()

    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()

        # Clear cache before each epoch
        torch.cuda.empty_cache()

        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device, scaler, USE_AMP,
            clear_cache_interval=CLEAR_CACHE_INTERVAL
        )

        # Clear cache before validation
        torch.cuda.empty_cache()

        val_loss, val_acc = validate(
            model, val_loader, criterion, device
        )

        scheduler.step()

        epoch_time = time.time() - epoch_start

        training_history['train_loss'].append(train_loss)
        training_history['train_acc'].append(train_acc)
        training_history['val_loss'].append(val_loss)
        training_history['val_acc'].append(val_acc)

        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} ({epoch_time:.1f}s)")
        print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
        print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

        # Save checkpoint for this epoch
        epoch_checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'training_history': training_history,
        }

        epoch_path = os.path.join(checkpoints_dir, f'model_epoch_{epoch+1}.pth')
        torch.save(epoch_checkpoint, epoch_path)
        print(f"  💾 Checkpoint saved: model_epoch_{epoch+1}.pth")

        # Save best model (separate from epoch checkpoints)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
                'train_acc': train_acc,
                'train_loss': train_loss,
            }
            torch.save(best_checkpoint, os.path.join(OUTPUT_DIR, 'best_model.pth'))
            print(f"  ✓ Best model updated! (Val Acc: {val_acc:.2f}%)")

    total_time = time.time() - start_time

    print("\n" + "=" * 80)
    print("TRAINING COMPLETE!")
    print("=" * 80)
    print(f"Total time: {total_time/60:.1f} minutes")
    print(f"Best validation accuracy: {best_val_acc:.2f}%")

    # Save corrupted files encountered during training
    # Handle both Subset and regular dataset
    base_dataset = dataset.dataset if isinstance(dataset, Subset) else dataset
    if hasattr(base_dataset, 'corrupted_indices') and len(base_dataset.corrupted_indices) > 0:
        print(f"\n⚠️  Corrupted files encountered: {len(base_dataset.corrupted_indices)}")
        corrupted_files = [base_dataset.file_names.iloc[i] for i in base_dataset.corrupted_indices]
        corrupted_annotations = base_dataset.annotations[
            base_dataset.annotations['file_name'].isin(corrupted_files)
        ][['Female', 'file_name']]

        skipped_path = os.path.join(BASE_DIR, 'annotations_skipped_in_training.csv')
        corrupted_annotations.to_csv(skipped_path, index=False)
        print(f"   ✓ Skipped files saved to: {skipped_path}")

    # Save results
    torch.save({
        'model_state_dict': model.state_dict(),
        'training_history': training_history,
    }, os.path.join(OUTPUT_DIR, 'final_model.pth'))

    history_df = pd.DataFrame(training_history)
    history_df.to_csv(os.path.join(OUTPUT_DIR, 'training_history.csv'), index=False)

    print(f"\n✓ All outputs saved to: {OUTPUT_DIR}")
    print("=" * 80)


if __name__ == '__main__':
    main()

# Resuming Training from epoch 20

In [ ]:
"""
OPTIMIZED ST-GCN Pipeline — with Resume from Checkpoint
Set RESUME_FROM_CHECKPOINT = True to continue from where you left off.
Set RESUME_FROM_CHECKPOINT = False to train from scratch.
"""

import os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch

# Check if CUDA (GPU support) is available
if torch.cuda.is_available():
    print("CUDA is available! Using GPU.")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("CUDA is not available. Using CPU.")

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from tqdm import tqdm
import time

# ============================================================================
# GRAPH & MODEL
# ============================================================================

class Graph:
    """Skeleton graph structure for human body (18 joints OpenPose format)"""
    def __init__(self, num_node=18, layout='openpose'):
        self.num_node = num_node
        self.layout = layout
        self.get_edge()
        self.get_adjacency()

    def get_edge(self):
        self.self_link = [(i, i) for i in range(self.num_node)]

        if self.num_node == 18:
            self.neighbor_link = [
                (0, 1), (1, 2), (2, 3), (3, 4), (1, 5), (5, 6), (6, 7),
                (1, 8), (8, 9), (9, 10), (10, 11), (8, 12), (12, 13),
                (13, 14), (0, 15), (15, 17), (0, 16), (16, 18),
            ]
        else:
            self.neighbor_link = [(i, i+1) for i in range(self.num_node-1)]

        self.edge = self.self_link + self.neighbor_link
        self.center = 1

    def get_adjacency(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edge:
            if i < self.num_node and j < self.num_node:
                A[i, j] = 1
                A[j, i] = 1

        A_with_self = A + np.eye(self.num_node)
        D = np.sum(A_with_self, axis=1)
        D_inv_sqrt = np.power(D, -0.5)
        D_inv_sqrt[np.isinf(D_inv_sqrt)] = 0
        D_mat = np.diag(D_inv_sqrt)
        A_normalized = D_mat @ A_with_self @ D_mat

        self.A = np.stack([np.eye(self.num_node), A_normalized, A_normalized])


class ConvTemporalGraphical(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv2d(in_channels, out_channels * kernel_size, kernel_size=1)

    def forward(self, x, A):
        x = self.conv(x)
        n, kc, t, v = x.size()
        x = x.view(n, self.kernel_size, kc // self.kernel_size, t, v)
        x = torch.einsum('nkctv,kvw->nctw', (x, A))
        return x.contiguous(), A


class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 dropout=0.2, residual=True):
        super().__init__()

        assert len(kernel_size) == 2
        assert kernel_size[0] % 2 == 1
        padding = ((kernel_size[0] - 1) // 2, 0)

        self.gcn = ConvTemporalGraphical(in_channels, out_channels, kernel_size[1])

        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, (kernel_size[0], 1),
                     (stride, 1), padding),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout, inplace=True),
        )

        if not residual:
            self.residual = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, A):
        res = self.residual(x)
        x, A = self.gcn(x, A)
        x = self.tcn(x) + res
        return self.relu(x), A


class GaitSTGCN(nn.Module):
    def __init__(self, in_channels=3, num_class=1, num_joints=18,
                 edge_importance_weighting=True, dropout=0.3):
        super().__init__()

        self.graph = Graph(num_node=num_joints)
        A = torch.tensor(self.graph.A, dtype=torch.float32, requires_grad=False)
        self.register_buffer('A', A)

        spatial_kernel_size = A.size(0)
        temporal_kernel_size = 9
        kernel_size = (temporal_kernel_size, spatial_kernel_size)

        self.data_bn = nn.BatchNorm1d(in_channels * A.size(1))

        self.st_gcn_networks = nn.ModuleList([
            STGCNBlock(in_channels, 64, kernel_size, 1, residual=False, dropout=0),
            STGCNBlock(64, 64, kernel_size, 1, dropout=dropout),
            STGCNBlock(64, 64, kernel_size, 1, dropout=dropout),
            STGCNBlock(64, 128, kernel_size, 2, dropout=dropout),
            STGCNBlock(128, 128, kernel_size, 1, dropout=dropout),
            STGCNBlock(128, 256, kernel_size, 2, dropout=dropout),
            STGCNBlock(256, 256, kernel_size, 1, dropout=dropout),
        ])

        if edge_importance_weighting:
            self.edge_importance = nn.Parameter(
                torch.ones(len(self.st_gcn_networks), *A.size())
            )
        else:
            self.edge_importance = [1] * len(self.st_gcn_networks)

        self.fc = nn.Linear(256, num_class)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        N, C, T, V, M = x.size()

        x = x.permute(0, 4, 3, 1, 2).contiguous()
        x = x.view(N * M, V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T)
        x = x.permute(0, 1, 3, 4, 2).contiguous()
        x = x.view(N * M, C, T, V)

        for gcn, importance in zip(self.st_gcn_networks, self.edge_importance):
            x, _ = gcn(x, self.A * importance)

        x = F.avg_pool2d(x, x.size()[2:])
        x = x.view(N, M, -1, 1, 1).mean(dim=1)
        x = x.view(N, -1)

        x = self.fc(x)
        return x

# ============================================================================
# PREPROCESSING
# ============================================================================

def detect_and_convert_format(data, target_V=18, target_C=3):
    shape = data.shape

    if len(shape) == 3:
        T, V, C = shape
        if V == target_V and C == target_C:
            return data
        elif V == target_V and C == 2 and target_C == 3:
            confidence = np.ones((T, V, 1), dtype=data.dtype)
            return np.concatenate([data, confidence], axis=2)
        flat = data.reshape(T, -1)
        return flat.reshape(T, target_V, target_C)

    elif len(shape) == 2:
        T, VC = shape
        if VC == target_V * target_C:
            return data.reshape(T, target_V, target_C)
        elif VC == target_V * 2 and target_C == 3:
            data_2d = data.reshape(T, target_V, 2)
            confidence = np.ones((T, target_V, 1), dtype=data.dtype)
            return np.concatenate([data_2d, confidence], axis=2)
        else:
            raise ValueError(f"Cannot convert shape {shape} to ({T}, {target_V}, {target_C})")
    else:
        raise ValueError(f"Unexpected data shape: {shape}")


def normalize_pose(data, method='center'):
    if method == 'center':
        center_idx = 1 if data.shape[1] >= 18 else 0
        center = data[:, center_idx:center_idx+1, :2]
        data_norm = data.copy()
        data_norm[:, :, :2] = data[:, :, :2] - center
        return data_norm
    return data


def handle_missing_values(data):
    for v in range(data.shape[1]):
        for c in range(data.shape[2]):
            series = data[:, v, c]
            nonzero = np.where(series != 0)[0]

            if len(nonzero) > 1:
                zero_indices = np.where(series == 0)[0]
                if len(zero_indices) > 0:
                    data[zero_indices, v, c] = np.interp(
                        zero_indices, nonzero, series[nonzero]
                    )
    return data

# ============================================================================
# DATASET
# ============================================================================

class GaitDatasetSTGCN(Dataset):
    def __init__(self, annotations_path, poses_dir, num_joints=18, num_coords=3,
                 normalize=True, handle_missing=True, skip_validation=True):
        self.annotations = pd.read_csv(annotations_path)
        self.poses_dir = poses_dir
        self.num_joints = num_joints
        self.num_coords = num_coords
        self.normalize = normalize
        self.handle_missing = handle_missing
        self.corrupted_indices = set()
        self.skip_validation = skip_validation

        self.file_names = self.annotations['file_name']
        self.labels = (self.annotations['Female'] > 0.5).astype(int).values

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        if idx in self.corrupted_indices:
            return self.__getitem__((idx + 1) % len(self))

        try:
            pose_path = os.path.join(self.poses_dir, self.file_names.iloc[idx])
            data = np.load(pose_path)

            if data.size == 0:
                raise ValueError("Empty file")

            data = detect_and_convert_format(data, self.num_joints, self.num_coords)

            if self.handle_missing:
                data = handle_missing_values(data)
            if self.normalize:
                data = normalize_pose(data, method='center')

            pose_sequence = torch.from_numpy(data).float()
            pose_sequence = pose_sequence.permute(2, 0, 1).unsqueeze(-1)
            label = torch.tensor(self.labels[idx]).float()

            return pose_sequence, label

        except Exception as e:
            self.corrupted_indices.add(idx)
            return self.__getitem__((idx + 1) % len(self))


def collate_fn_stgcn(batch, max_seq_length=300):
    sequences, labels = zip(*batch)

    sequences = [seq[:, :max_seq_length, :, :] if seq.shape[1] > max_seq_length else seq
                 for seq in sequences]

    max_T = max(seq.shape[1] for seq in sequences)

    padded_sequences = []
    lengths = []

    for seq in sequences:
        C, T, V, M = seq.shape
        lengths.append(T)

        if T < max_T:
            padding = torch.zeros(C, max_T - T, V, M)
            seq = torch.cat([seq, padding], dim=1)

        padded_sequences.append(seq)

    padded_sequences = torch.stack(padded_sequences, dim=0)
    labels = torch.stack(labels, dim=0)

    return padded_sequences, labels, torch.tensor(lengths)

# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================

def train_epoch(model, dataloader, criterion, optimizer, device, scaler,
                use_amp=True, clear_cache_interval=25):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Training", leave=False)
    for batch_idx, (poses, labels, lengths) in enumerate(pbar):
        poses = poses.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast('cuda'):
                outputs = model(poses)
                loss = criterion(outputs.squeeze(), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(poses)
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        if (batch_idx + 1) % clear_cache_interval == 0:
            torch.cuda.empty_cache()

        pbar.set_postfix({
            'loss': f'{total_loss/(batch_idx+1):.4f}',
            'acc': f'{100*correct/total:.2f}%'
        })

    return total_loss / len(dataloader), 100 * correct / total


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for poses, labels, lengths in tqdm(dataloader, desc="Validating", leave=False):
            poses = poses.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(poses)
            loss = criterion(outputs.squeeze(), labels)

            total_loss += loss.item()
            predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(dataloader), 100 * correct / total

# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    print("=" * 80)
    print("OPTIMIZED ST-GCN GENDER CLASSIFICATION")
    print("=" * 80)

    # ========================================================================
    # CONFIG
    # ========================================================================
    BASE_DIR       = '/content/drive/MyDrive/Gait Project/DenseGait'
    ANNOTATIONS_PATH = os.path.join(BASE_DIR, 'annotations_present.csv')
    POSES_DIR      = os.path.join(BASE_DIR, 'poses')
    OUTPUT_DIR     = os.path.join(BASE_DIR, 'output')

    # ── RESUME SETTINGS ──────────────────────────────────────────────────────
    RESUME_FROM_CHECKPOINT = True   # ← True = continue training, False = start fresh
    RESUME_EPOCH           = 20     # ← the last epoch number you completed
    ADDITIONAL_EPOCHS      = 30     # ← how many MORE epochs to run (total = 50)
    # ─────────────────────────────────────────────────────────────────────────

    USE_SUBSET         = True
    SUBSET_SIZE        = 30000
    BATCH_SIZE         = 64
    NUM_WORKERS        = 2
    USE_AMP            = True
    SKIP_VALIDATION    = True
    PREFETCH_FACTOR    = 2
    CLEAR_CACHE_INTERVAL = 25
    MAX_SEQUENCE_LENGTH  = 300
    NUM_JOINTS         = 18
    NUM_COORDS         = 3
    LEARNING_RATE      = 0.001
    WEIGHT_DECAY       = 0.0001
    DROPOUT            = 0.3

    # Total epochs = previous + additional when resuming, else just ADDITIONAL_EPOCHS
    NUM_EPOCHS = (RESUME_EPOCH + ADDITIONAL_EPOCHS) if RESUME_FROM_CHECKPOINT else ADDITIONAL_EPOCHS

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ========================================================================
    # 1. LOAD DATASET
    # ========================================================================
    print("\n1. Loading dataset...")
    print(f"   Using: {ANNOTATIONS_PATH}")

    if not os.path.exists(ANNOTATIONS_PATH):
        print(f"\n   ERROR: {ANNOTATIONS_PATH} not found!")
        print("   Run the file splitting script first to create annotations_present.csv")
        return

    dataset = GaitDatasetSTGCN(
        annotations_path=ANNOTATIONS_PATH,
        poses_dir=POSES_DIR,
        num_joints=NUM_JOINTS,
        num_coords=NUM_COORDS,
        normalize=True,
        handle_missing=True,
        skip_validation=SKIP_VALIDATION
    )
    print(f"   Total available samples: {len(dataset):,}")

    if SKIP_VALIDATION:
        print("   ⚡ Skipping pre-validation (errors handled during training)")

    if USE_SUBSET and len(dataset) > SUBSET_SIZE:
        indices = torch.randperm(len(dataset))[:SUBSET_SIZE].tolist()
        dataset = Subset(dataset, indices)
        print(f"   Using subset: {len(dataset):,} samples")

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(
        dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    print(f"   Training samples:   {train_size:,}")
    print(f"   Validation samples: {val_size:,}")

    # ========================================================================
    # 2. CREATE DATALOADERS
    # ========================================================================
    print("\n2. Creating dataloaders...")

    def collate_wrapper(batch):
        return collate_fn_stgcn(batch, max_seq_length=MAX_SEQUENCE_LENGTH)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_wrapper,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_wrapper,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
    )
    print(f"   Batch size: {BATCH_SIZE}")
    print(f"   Batches per epoch: {len(train_loader)}")
    print(f"   Workers: {NUM_WORKERS}, Prefetch: {PREFETCH_FACTOR}")
    print(f"   Max sequence length: {MAX_SEQUENCE_LENGTH}")

    # ========================================================================
    # 3. INITIALIZE MODEL
    # ========================================================================
    print("\n3. Initializing model...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"   Device: {device}")

    if device.type == 'cpu':
        print("   WARNING: Running on CPU. Training will be SLOW!")
        print("   Go to Runtime > Change runtime type > Select GPU")

    model = GaitSTGCN(
        in_channels=NUM_COORDS,
        num_class=1,
        num_joints=NUM_JOINTS,
        edge_importance_weighting=True,
        dropout=DROPOUT
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"   Total parameters: {total_params:,}")

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=5, gamma=0.5
    )
    scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

    if USE_AMP:
        print("   Mixed precision: ENABLED (2x speedup)")

    # ── Resume from checkpoint ────────────────────────────────────────────────
    start_epoch      = 0
    best_val_acc     = 0.0
    training_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    if RESUME_FROM_CHECKPOINT:
        ckpt_path = os.path.join(OUTPUT_DIR, 'checkpoints', f'model_epoch_{RESUME_EPOCH}.pth')

        if not os.path.exists(ckpt_path):
            print(f"\n   ERROR: Checkpoint not found at {ckpt_path}")
            print(f"   Make sure model_epoch_{RESUME_EPOCH}.pth exists in {OUTPUT_DIR}/checkpoints/")
            print("   Set RESUME_FROM_CHECKPOINT = False to train from scratch instead.")
            return

        print(f"\n   Loading checkpoint: model_epoch_{RESUME_EPOCH}.pth")
        checkpoint = torch.load(ckpt_path, map_location=device)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        training_history = checkpoint['training_history']
        start_epoch      = checkpoint['epoch']                            # = 20
        best_val_acc     = max(checkpoint['training_history']['val_acc']) # best so far

        print(f"   ✓ Resumed from epoch {start_epoch}")
        print(f"   ✓ Best val acc so far: {best_val_acc:.2f}%")
        print(f"   ✓ Will train epochs {start_epoch + 1} → {NUM_EPOCHS}")
        print(f"   ✓ Current LR: {scheduler.get_last_lr()}")
    else:
        print(f"\n   Starting fresh — will train epochs 1 → {NUM_EPOCHS}")
    # ─────────────────────────────────────────────────────────────────────────

    # ========================================================================
    # 4. TRAINING LOOP
    # ========================================================================
    print("\n4. Training...")
    print("-" * 80)

    checkpoints_dir = os.path.join(OUTPUT_DIR, 'checkpoints')
    os.makedirs(checkpoints_dir, exist_ok=True)

    start_time = time.time()

    # Loop starts at start_epoch (0 for fresh, 20 for resumed)
    for epoch in range(start_epoch, NUM_EPOCHS):
        epoch_start = time.time()

        torch.cuda.empty_cache()

        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device, scaler, USE_AMP,
            clear_cache_interval=CLEAR_CACHE_INTERVAL
        )

        torch.cuda.empty_cache()

        val_loss, val_acc = validate(
            model, val_loader, criterion, device
        )

        scheduler.step()

        epoch_time = time.time() - epoch_start

        training_history['train_loss'].append(train_loss)
        training_history['train_acc'].append(train_acc)
        training_history['val_loss'].append(val_loss)
        training_history['val_acc'].append(val_acc)

        # epoch+1 correctly shows 21, 22 ... when resumed from epoch 20
        print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS} ({epoch_time:.1f}s)")
        print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
        print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")
        print(f"  LR: {scheduler.get_last_lr()[0]:.2e}")

        # Save per-epoch checkpoint
        epoch_checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'training_history': training_history,
        }

        epoch_path = os.path.join(checkpoints_dir, f'model_epoch_{epoch + 1}.pth')
        torch.save(epoch_checkpoint, epoch_path)
        print(f"  💾 Checkpoint saved: model_epoch_{epoch + 1}.pth")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
                'train_acc': train_acc,
                'train_loss': train_loss,
            }
            torch.save(best_checkpoint, os.path.join(OUTPUT_DIR, 'best_model.pth'))
            print(f"  ✓ Best model updated! (Val Acc: {val_acc:.2f}%)")

    total_time = time.time() - start_time

    print("\n" + "=" * 80)
    print("TRAINING COMPLETE!")
    print("=" * 80)
    print(f"Total time this run: {total_time/60:.1f} minutes")
    print(f"Best validation accuracy: {best_val_acc:.2f}%")
    print(f"Epochs completed: {start_epoch + 1} → {NUM_EPOCHS}")

    # Save corrupted files list
    base_dataset = dataset.dataset if isinstance(dataset, Subset) else dataset
    if hasattr(base_dataset, 'corrupted_indices') and len(base_dataset.corrupted_indices) > 0:
        print(f"\n⚠️  Corrupted files encountered: {len(base_dataset.corrupted_indices)}")
        corrupted_files = [base_dataset.file_names.iloc[i] for i in base_dataset.corrupted_indices]
        corrupted_annotations = base_dataset.annotations[
            base_dataset.annotations['file_name'].isin(corrupted_files)
        ][['Female', 'file_name']]
        skipped_path = os.path.join(BASE_DIR, 'annotations_skipped_in_training.csv')
        corrupted_annotations.to_csv(skipped_path, index=False)
        print(f"   ✓ Skipped files saved to: {skipped_path}")

    # Save final model + full history (all epochs, including pre-resume)
    torch.save({
        'model_state_dict': model.state_dict(),
        'training_history': training_history,
    }, os.path.join(OUTPUT_DIR, 'final_model.pth'))

    history_df = pd.DataFrame(training_history)
    history_df.to_csv(os.path.join(OUTPUT_DIR, 'training_history.csv'), index=False)

    print(f"\n✓ All outputs saved to: {OUTPUT_DIR}")
    print(f"  - training_history.csv  (all {len(history_df)} epochs)")
    print(f"  - final_model.pth")
    print(f"  - best_model.pth        (epoch {history_df['val_acc'].idxmax() + 1}, val acc {history_df['val_acc'].max():.2f}%)")
    print("=" * 80)


if __name__ == '__main__':
    main()